In [1]:
# 1. Import & Load
import pandas as pd
import numpy as np

df = pd.read_excel("C:/Users/mdyus/Downloads/Retail_Merchandising_Analysis/Data/Cleaned/online_retail_cleaned.xlsx")
df.head(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [2]:
# 2. Standardize Column Names
df.columns = [
    'invoice_id',
    'product_id',
    'description',
    'quantity',
    'invoice_date',
    'unit_price',
    'customer_id',
    'country'
]

In [3]:
# 3. Drop Rows Where Product Info is Missing
df = df.dropna(subset=['description'])

In [4]:
# 4. Remove Returns / Invalid Rows
df = df[df['quantity'] > 0]
df = df[df['unit_price'] > 0]

In [5]:
# 5. customer_id Cleanup
df['customer_id'] = df['customer_id'].astype('Int64')
df['customer_id'] = df['customer_id'].astype('string').fillna('Anonymous')

In [6]:
# 6. Create Sales (Revenue) Column
df['sales'] = df['quantity'] * df['unit_price']

In [7]:
# 7. Category Mapping

# STEP 1: Main Category Mapping
def map_category(desc):
    desc = str(desc).lower()

    if any(x in desc for x in ['set', 'kit', 'tool', 'scissor', 'knife']):
        return 'Tools & Utility'
    elif any(x in desc for x in ['lamp', 'light', 'candle', 'lantern']):
        return 'Lighting'
    elif any(x in desc for x in ['mug', 'cup', 'plate', 'bowl', 'kitchen']):
        return 'Kitchen & Dining'
    elif any(x in desc for x in ['bag', 'box', 'storage', 'basket']):
        return 'Storage & Organization'
    elif any(x in desc for x in ['decor', 'frame', 'art', 'ornament']):
        return 'Home Decor'
    elif any(x in desc for x in ['christmas', 'gift', 'card', 'wrap']):
        return 'Seasonal & Gifts'
    elif any(x in desc for x in ['garden', 'plant', 'outdoor']):
        return 'Outdoor'
    elif any(x in desc for x in ['toy', 'game', 'kids']):
        return 'Toys & Kids'
    elif any(x in desc for x in ['textile', 'cloth', 'towel', 'fabric']):
        return 'Home Textiles'
    elif any(x in desc for x in ['paper', 'stationery', 'note', 'pen']):
        return 'Stationery'
    else:
        return 'Miscellaneous'

df['category'] = df['description'].apply(map_category)


# STEP 2: Refinement for Miscellaneous
def refine_misc(desc):
    desc = str(desc).lower()

    if any(x in desc for x in ['cake', 'tea', 'bottle']):
        return 'Kitchen & Dining'
    elif any(x in desc for x in ['metal', 'tin', 'sign', 'vintage', 'retro']):
        return 'Home Decor'
    elif any(x in desc for x in ['door', 'mat']):
        return 'Home Utility'
    elif any(x in desc for x in ['feltcraft', 'fairy']):
        return 'Toys & Kids'
    elif any(x in desc for x in ['case', 'pack']):
        return 'Storage & Organization'
    else:
        return 'Miscellaneous'

mask = df['category'] == 'Miscellaneous'
df.loc[mask, 'category'] = df.loc[mask, 'description'].apply(refine_misc)


# STEP 3: Validation
print(df['category'].value_counts(normalize=True))

category
Home Decor                0.209011
Miscellaneous             0.193381
Storage & Organization    0.141923
Tools & Utility           0.122901
Kitchen & Dining          0.118665
Lighting                  0.067618
Seasonal & Gifts          0.049044
Stationery                0.032842
Home Utility              0.025117
Toys & Kids               0.018494
Outdoor                   0.014375
Home Textiles             0.006629
Name: proportion, dtype: float64


In [8]:
# 8. Vendor Simulation
vendors = [
    'Stanley Black & Decker',
    'Bosch',
    'Makita',
    'DeWalt',
    '3M',
    'Sherwin-Williams',
    'Whirlpool',
    'GE Appliances',
    'Andersen',
    'Delta Faucet'
]

df['vendor'] = np.random.choice(vendors, size=len(df))

In [9]:
# 9. Cost + Margin Engineering
df['cost'] = df['unit_price'] * np.random.uniform(0.55, 0.72, size=len(df))
df['profit'] = df['sales'] - df['cost']
df['margin_pct'] = df['profit'] / df['sales']

In [10]:
# 10. Channel Assignment
df['channel'] = np.random.choice(
    ['In-Store', 'Online'],
    size=len(df),
    p=[0.7, 0.3]
)

In [11]:
# 11. Promotion Flag
category_avg = df.groupby('category')['quantity'].transform('mean')

df['promo_flag'] = np.where(
    df['quantity'] > 1.5 * category_avg,
    'Promo',
    'Regular'
)

In [12]:
# 12. Vendor Fill Rate
vendor_fill = {
    v: np.random.uniform(0.72, 0.97)
    for v in df['vendor'].unique()
}

df['fill_rate'] = df['vendor'].map(vendor_fill)

In [13]:
# 13. Time-Based Features
df['year'] = df['invoice_date'].dt.year
df['month'] = df['invoice_date'].dt.month
df['week'] = df['invoice_date'].dt.isocalendar().week
df['day'] = df['invoice_date'].dt.day
df['weekday'] = df['invoice_date'].dt.day_name()

In [14]:
# 14. Product Performance + Margin Flags

# Revenue flag
product_sales = df.groupby('product_id')['sales'].transform('sum')
df['high_revenue_product'] = product_sales > product_sales.quantile(0.75)

# Volume flag
product_units = df.groupby('product_id')['quantity'].transform('sum')
df['high_volume_product'] = product_units > product_units.quantile(0.75)

# Margin flags
df['low_margin_flag'] = df['margin_pct'] < 0.20
df['high_margin_flag'] = df['margin_pct'] > 0.40

In [15]:
# 15. Customer Segmentation
customer_spend = df.groupby('customer_id')['sales'].transform('sum')

df['customer_segment'] = pd.qcut(
    customer_spend,
    q=3,
    labels=['Low', 'Medium', 'High']
)

In [16]:
# 16. Invoice-Level Metrics (Correct Granularity)

# Recalculate at invoice level
invoice_metrics = (
    df.groupby('invoice_id')
      .agg(
          order_value=('sales', 'sum'),
          items_per_order=('quantity', 'sum')
      )
      .reset_index()
)

# Merge correct values back
df = df.drop(columns=['order_value', 'items_per_order'], errors='ignore')
df = df.merge(invoice_metrics, on='invoice_id', how='left')

# Category contribution (row-level vs invoice-level)
df['category_contribution'] = df['sales'] / df['order_value']
df['category_contribution'] = df['category_contribution'].replace([np.inf, -np.inf], np.nan).fillna(0)

# Channel efficiency
df['revenue_per_unit'] = df['sales'] / df['quantity']

In [17]:
# 17. Rounding

# Financial metrics
df['sales'] = df['sales'].round(2)
df['cost'] = df['cost'].round(2)
df['profit'] = df['profit'].round(2)

# Percentages and ratios
df['margin_pct'] = df['margin_pct'].round(2)
df['fill_rate'] = df['fill_rate'].round(2)
df['revenue_per_unit'] = df['revenue_per_unit'].round(2)

In [21]:
# 18. Final Data Validation for SQL Layer

def clean_df(df):

    # Strip whitespace + replace blanks with null
    df = df.apply(lambda x: x.astype(str).str.strip() if x.dtype == 'object' else x)
    df.replace(r'^\s*$', np.nan, regex=True, inplace=True)

    # Date
    df['invoice_date'] = pd.to_datetime(df['invoice_date'], errors='coerce', dayfirst=True)
    df = df.dropna(subset=['invoice_date'])

    # Numeric
    num = ['quantity', 'unit_price', 'cost', 'sales', 'profit', 'margin_pct',
           'fill_rate', 'order_value', 'items_per_order', 'category_contribution', 'revenue_per_unit']
    df[num] = df[num].apply(pd.to_numeric, errors='coerce')

    # Integer columns
    ints = ['year', 'month', 'week', 'day']
    df[ints] = df[ints].apply(pd.to_numeric, errors='coerce')

    # Boolean columns → integer (0/1)
    bools = ['high_revenue_product', 'high_volume_product', 'low_margin_flag', 'high_margin_flag']
    df[bools] = df[bools].astype(str).apply(
        lambda x: x.str.lower().map({'true': 1, 'false': 0, '1': 1, '0': 0})
    ).fillna(0)

    # ID columns
    for c in ['invoice_id', 'product_id', 'customer_id']:
        df[c] = df[c].astype(str).str.replace('.0', '', regex=False)

    df['customer_id'] = df['customer_id'].replace({'nan': 'Anonymous', '': 'Anonymous', 'NA': 'Anonymous', '<NA>': 'Anonymous'})

    # Drop rows with missing critical values
    df = df.dropna(subset=['quantity', 'unit_price'] + ints)

    # Deduplicate
    df = df.sort_values('invoice_date').drop_duplicates(['invoice_id', 'product_id'], keep='last')

    # Final type enforcement
    df[ints + ['quantity', 'items_per_order']] = df[ints + ['quantity', 'items_per_order']].astype(int)
    df[bools] = df[bools].astype(int)
    df[df.select_dtypes('float').columns] = df.select_dtypes('float').round(2)

    return df.reset_index(drop=True)


df = clean_df(df)

In [22]:
# 19. Export Clean File
df.to_csv(
    "C:/Users/mdyus/Downloads/Retail_Merchandising_Analysis/Data/Cleaned/clean_data.csv",
    index=False
)

print(f"Export complete. Shape: {df.shape}")

Export complete. Shape: (498373, 31)


In [23]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 498373 entries, 0 to 498372
Data columns (total 31 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   invoice_id             498373 non-null  str           
 1   product_id             498373 non-null  str           
 2   description            498373 non-null  str           
 3   quantity               498373 non-null  int64         
 4   invoice_date           498373 non-null  datetime64[us]
 5   unit_price             498373 non-null  float64       
 6   customer_id            498373 non-null  str           
 7   country                498373 non-null  str           
 8   sales                  498373 non-null  float64       
 9   category               498373 non-null  str           
 10  vendor                 498373 non-null  str           
 11  cost                   498373 non-null  float64       
 12  profit                 498373 non-null  float64       
